In [ ]:
import pandas as pd
import arviz as az
from IPython.display import display
from os import listdir as ls
import pickle
import pycountry
from IPython.display import display, Markdown

from emu_renewal.constants import BASE_PATH, ANALYSIS_NAMES
from emu_renewal.outputs import load_targets
from emu_renewal.plotting import plot_input_recovery

# Purpose
The following analyses are intended to show the models ability to
recover key known parameters through its calibration algorithm.

# Analysis steps
The steps of this analysis are as follows:

1. One country was selected from the countries included in the main analysis other than those from Oceania (without replacement on iterations)
2. One mobility approach was selected from the types of mobility analysis that were run for that country
3. One accepted parameter set was randomly selected from that country-mobility type calibration
4. Only the targets for deaths and cases were identified from this analysis
5. A calibration was run for that country-mobility type combination, with all parameters fixed at the values selected in Step 3 - except for the variable process and the mobility exponent mapping parameter (if mobility was included), which were calibrated
6. The results were plotted as below
The upper panels of figures below compare the calibration fit to data for cases and deaths. The lower-left panel compares the target values for the variable process from the base case analysis (red circles) against the variable process estimates from this re-calibration analysis (black lines). The lower-right panel compares the target value for the mobility exponent mapping parameter (vertical blue bar) against the estimated posterior density for this re-calibration analysis (red area).

\newpage
# Analysis results

In [ ]:
analysis_time = "20260203_1740"
job_path = BASE_PATH / "identify_outputs" / analysis_time
countries = ls(job_path)
for iso3 in countries:
    country = pycountry.countries.lookup(iso3).name
    mob_source = ls(job_path / iso3)[0]
    display(Markdown(f"## {country}, {ANALYSIS_NAMES[mob_source]}"))
    path = job_path / iso3 / mob_source
    idata = az.from_netcdf( path/ "idata_filtered.nc")
    targets = load_targets(path)
    spaghetti = pd.read_hdf(path / "spaghetti.h5")
    updates = pd.read_hdf(path / "updates.h5")
    scalar_params = pickle.load(open(path / "scalar_params.pkl", "rb"))
    multi_params = pickle.load(open(path / "multi_params.pkl", "rb"))    
    display(plot_input_recovery(scalar_params, multi_params, idata, targets, spaghetti, updates))